# Week 4 — Fused vs unfused attention + Triton SGEMM

Single-head SDPA (`QKᵀ` → softmax → `·V`), `head_dim=64`, `seq_len` 256/512/1024.

- **Unfused:** three kernels, `seq×seq` scores in global memory
- **Fused:** one kernel, online softmax, K/V tiles in shared memory (baby FlashAttention)
- Then **Triton** tiled SGEMM vs `torch.matmul` (cuBLAS), and PyTorch `scaled_dot_product_attention` as a reference

**Runtime → Change runtime type → T4 GPU**. Paste every table back into chat. Do not fill the README yourself.

In [ ]:
import shutil, subprocess, sys
if shutil.which("nvidia-smi") is None:
    sys.exit("No GPU. Runtime → Change runtime type → T4 GPU.")
print(subprocess.check_output(["nvidia-smi"], text=True))
print("GPU:", subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip())

In [ ]:
%%bash
set -euo pipefail
REPO_URL="https://github.com/preethamdandu/cuda-kernel-optimization.git"
DIR="cuda-kernel-optimization"
if [ -d "$DIR/.git" ]; then
  cd "$DIR" && git pull --ff-only
else
  git clone "$REPO_URL" "$DIR"
fi

In [ ]:
%%bash
set -euo pipefail
if ! command -v nvidia-smi >/dev/null || ! command -v nvcc >/dev/null; then
  echo "No GPU / no nvcc. Runtime → Change runtime type → T4 GPU." >&2
  exit 1
fi
cd cuda-kernel-optimization
ARCH=sm_$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d '.' | tr -d ' ')
echo "building for $ARCH"
nvcc -O3 -arch=$ARCH -lineinfo \
  benchmark/bench_attn.cu src/07_attention_unfused.cu src/08_attention_fused.cu \
  -o bench_attn
echo "build ok"
./bench_attn --seqs 256 512 1024

In [ ]:
import time
import torch
import torch.nn.functional as F

print("GPU:", torch.cuda.get_device_name(0))
print("flash_sdp:", torch.backends.cuda.flash_sdp_enabled())
print("mem_efficient_sdp:", torch.backends.cuda.mem_efficient_sdp_enabled())
print("math_sdp:", torch.backends.cuda.math_sdp_enabled())

HEAD = 64
ITERS = 10
WARMUP = 3

def bench(fn):
    for _ in range(WARMUP):
        fn()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(ITERS):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) * 1e3 / ITERS

print("T4 is sm_75; FlashAttention CUDA is typically sm_80+. MATH / MEM_EFFICIENT are the fair refs.")
print(f"{'seq':>6} {'sdpa_ms':>10} {'backend'}")

def time_backend(q, k, v, **sdp_flags):
    ctx = torch.backends.cuda.sdp_kernel(**sdp_flags)
    with ctx:
        return bench(lambda: F.scaled_dot_product_attention(q, k, v))

for seq in (256, 512, 1024):
    q = torch.randn(1, 1, seq, HEAD, device="cuda", dtype=torch.float32)
    k = torch.randn(1, 1, seq, HEAD, device="cuda", dtype=torch.float32)
    v = torch.randn(1, 1, seq, HEAD, device="cuda", dtype=torch.float32)
    math_ms = time_backend(q, k, v, enable_flash=False, enable_math=True, enable_mem_efficient=False)
    eff_ms = time_backend(q, k, v, enable_flash=False, enable_math=False, enable_mem_efficient=True)
    print(f"{seq:6d} {math_ms:10.4f}  MATH")
    print(f"{seq:6d} {eff_ms:10.4f}  MEM_EFFICIENT")

In [ ]:
%%bash
set -euo pipefail
cd cuda-kernel-optimization
python3 - <<'PY'
import triton, torch
print("triton", triton.__version__)
print("torch", torch.__version__)
PY
python3 triton/matmul.py --sizes 1024 2048 4096

Paste three things back into chat:

1. `./bench_attn` table (unfused vs fused)
2. PyTorch SDPA MATH / MEM_EFFICIENT times
3. Triton vs torch.matmul table

If the GPU name is not Tesla T4, say so before any README edit.